# Fase exploratoria: máscaras con SAM 2
## Cuaderno A: prompts desde la U-Net v1 y muestra de val

Este cuaderno es **exploratorio**: no modifica ningún artefacto de la v1 ni del módulo 01. Usa TensorFlow, así que corre **solo** y no comparte sesión con el cuaderno B (PyTorch).

Qué produce:
- `sam2/prompts.csv`: caja y centroide de la U-Net v1 para cada imagen (el prompt de SAM 2).
- `sam2/muestra_val.csv`: 30 imágenes de val en tres estratos (`aleatoria` 15, `peor` 8, `descartada` 7) para el piloto visual.

In [ ]:
"""
Autor: Jairo de Jesús Blanco Pérez
Codigo: <tu código>
Materia: Inteligencia Artificial 2026-2
Tema: Fase exploratoria: máscaras con SAM 2, A: prompts desde la U-Net v1 y muestra de val
"""

Ahora vamos a importar las librerías, montar Drive, fijar las rutas y verificar el MD5 de `splits.csv`.

In [ ]:
import os, json, hashlib, zipfile             # sistema de archivos, JSON, MD5 y zip
import numpy as np, pandas as pd, cv2         # arreglos, tablas y OpenCV
import matplotlib.pyplot as plt               # figuras
import tensorflow as tf                       # carga de la U-Net v1
from google.colab import drive                # acceso a Drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/sylfish'       # raíz de datos
P_SPLITS = f'{ROOT}/data/splits/splits.csv'   # partición oficial
PROC = f'{ROOT}/data/processed'               # mask_qc.csv y cachés .npz
P_QC = f'{PROC}/mask_qc.csv'                  # control de calidad de GrabCut
ZIP_IMG = f'{ROOT}/data/raw/SylFishBD_IMAGE.zip'   # imágenes originales (solo existe el zip)
DIR_IMG_LOCAL = '/content/sylfish_images'     # carpeta local de extracción
SEED = 42
MD5_SPLITS = '5b174a510317656566c44916ff8da4a7'

md5 = hashlib.md5(open(P_SPLITS, 'rb').read()).hexdigest()
assert md5 == MD5_SPLITS, f'MD5 distinto: {md5}'

# archivos que este cuaderno necesita
necesarios = [P_SPLITS, P_QC, ZIP_IMG, f'{ROOT}/models/unet_v1.keras', f'{ROOT}/models/unet_v1_umbral.json'] + \
             [f'{PROC}/unet_256_{s}.npz' for s in ('train', 'val', 'test')]
faltan = [f for f in necesarios if not os.path.exists(f)]
print('MD5 ok | faltan:', faltan)

Ahora vamos a extraer las imágenes originales a disco local. En Drive solo existe el zip `data/raw/SylFishBD_IMAGE.zip` (no hay carpeta `images/`). Indexamos por el nombre de archivo sin extensión (`stem`, único en el dataset), así no dependemos de cómo esté organizado el zip por dentro.

In [ ]:
marca = f'{DIR_IMG_LOCAL}/.listo'                                  # se escribe al terminar la extracción
if not os.path.exists(marca):                                       # extrae una sola vez por sesión (~420 MB)
    with zipfile.ZipFile(ZIP_IMG) as zf:
        zf.extractall(DIR_IMG_LOCAL)
    open(marca, 'w').close()
IMG_INDEX = {os.path.splitext(f)[0]: os.path.join(r, f)             # stem -> ruta local
             for r, _, fs in os.walk(DIR_IMG_LOCAL) for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png'))}

def ruta_img(fp):                                                   # ruta local de una imagen a partir de su filepath
    return IMG_INDEX[os.path.splitext(os.path.basename(fp))[0]]

print(len(IMG_INDEX), 'imágenes indexadas (esperado: 9075)')

Ahora vamos a cargar la U-Net v1, el umbral $\tau$ y las tablas. Todo se cruza por `filepath`, nunca por posición.

In [ ]:
model = tf.keras.models.load_model(f'{ROOT}/models/unet_v1.keras', compile=False)   # sin pérdida ni métricas personalizadas
tau = json.load(open(f'{ROOT}/models/unet_v1_umbral.json'))['tau']                  # 0.6
splits = pd.read_csv(P_SPLITS)
qc = pd.read_csv(P_QC)[['filepath', 'area_frac', 'motivo', 'qc_ok']]
print('tau =', tau, '| parámetros:', model.count_params())                          # esperado: 1,946,993
print(qc['motivo'].value_counts())                                                    # cómo se nombra la "vacía"

Ahora vamos a predecir por lotes. Por imagen:
1. ampliamos las **probabilidades** a $500 \times 500$ con interpolación bilineal;
2. binarizamos con $\tau$ ($m_{ij} = \mathbf{1}[p_{ij} > \tau]$);
3. nos quedamos con el contorno de mayor área (igual que la demo) y guardamos su caja y su centroide.

Además calculamos el IoU a $256 \times 256$ contra GrabCut, solo para comprobar que la v1 cargó bien: en val con `qc_ok` debe dar $\approx 0.833$.

In [ ]:
def prompts_split(split, lote=64):
    d = np.load(f'{PROC}/unet_256_{split}.npz', allow_pickle=True)
    X, M, fps = d['X'], d['M'], [str(f) for f in d['filepath']]
    assert set(fps) == set(splits.loc[splits.split == split, 'filepath'])    # mismo conjunto de imágenes
    filas, ious = [], []
    for i in range(0, len(X), lote):
        p = model.predict(X[i:i+lote].astype('float32') / 255.0, verbose=0)[..., 0]   # (n, 256, 256)
        for k in range(len(p)):
            # IoU a 256 contra GrabCut (verificación; sin contorno mayor, igual que el módulo 03)
            pred, ref = p[k] > tau, M[i+k] > 127
            ious.append((pred & ref).sum() / max((pred | ref).sum(), 1))
            # prompt: probabilidades a 500x500 -> tau -> contorno mayor
            m = (cv2.resize(p[k], (500, 500), interpolation=cv2.INTER_LINEAR) > tau).astype('uint8')
            cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cs:
                filas.append(dict(filepath=fps[i+k], split=split, ux=np.nan, uy=np.nan, uw=np.nan,
                                  uh=np.nan, ucx=np.nan, ucy=np.nan, unet_vacia=True))
                continue
            c = max(cs, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(c)
            lleno = np.zeros((500, 500), 'uint8'); cv2.drawContours(lleno, [c], -1, 1, -1)
            mo = cv2.moments(lleno, binaryImage=True)
            filas.append(dict(filepath=fps[i+k], split=split, ux=x, uy=y, uw=w, uh=h,
                              ucx=mo['m10']/mo['m00'], ucy=mo['m01']/mo['m00'], unet_vacia=False))
    df = pd.DataFrame(filas); df['iou256_v1'] = ious
    return df

prompts = pd.concat([prompts_split(s) for s in ('train', 'val', 'test')], ignore_index=True)
prompts = prompts.merge(qc[['filepath', 'qc_ok']], on='filepath', how='left')
v = prompts[(prompts.split == 'val') & prompts.qc_ok]
print(f'mIoU v1 en val qc_ok: {v.iou256_v1.mean():.4f}  (esperado ≈ 0.833, n = {len(v)})')

Ahora vamos a contar cuántas predicciones vacías hay por split y por clase. Si son muchas (más de $\approx 5\,\%$), esas imágenes usarán el prompt de respaldo P1 en el cuaderno B.

In [ ]:
prompts['clase'] = prompts.filepath.str.split('/').str[0]
print(prompts.groupby('split').unet_vacia.agg(['sum', 'mean']))
print(prompts.groupby('clase').unet_vacia.agg(['sum', 'mean']).round(3))

Ahora vamos a dibujar 12 cajas sobre sus imágenes de val y a revisarlas a ojo.

In [ ]:
vv = prompts[(prompts.split == 'val') & ~prompts.unet_vacia].sample(12, random_state=SEED)
fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for a, (_, r) in zip(ax.ravel(), vv.iterrows()):
    img = cv2.cvtColor(cv2.imread(ruta_img(r.filepath)), cv2.COLOR_BGR2RGB)   # OpenCV lee BGR
    x, y, w, h = map(int, (r.ux, r.uy, r.uw, r.uh))
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 3)
    a.imshow(img); a.set_title(r.filepath, fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

Ahora vamos a guardar `prompts.csv` con escritura atómica (archivo temporal y renombrado).

In [ ]:
os.makedirs(f'{ROOT}/sam2', exist_ok=True)
cols = ['filepath', 'split', 'ux', 'uy', 'uw', 'uh', 'ucx', 'ucy', 'unet_vacia', 'iou256_v1', 'qc_ok']
tmp = f'{ROOT}/sam2/prompts.csv.tmp'
prompts[cols].to_csv(tmp, index=False); os.replace(tmp, f'{ROOT}/sam2/prompts.csv')
print(len(prompts), 'filas guardadas')

Ahora vamos a sortear la muestra de val para el piloto visual, con semilla 42:
- `aleatoria` (15): de val con `qc_ok`, sin las de `peor`.
- `peor` (8): las de menor IoU de la v1 contra GrabCut en val con `qc_ok`.
- `descartada` (7): de val con `qc_ok = False`, 4 vacías y 3 de área grande.

El `id` (1 a 30) sigue el orden del archivo y es el mismo en las cuadrículas de los cuadernos B y C.

In [ ]:
vq = prompts[(prompts.split == 'val') & prompts.qc_ok].merge(qc[['filepath', 'area_frac']], on='filepath')
peor = vq.nsmallest(8, 'iou256_v1')
azar = vq[~vq.filepath.isin(peor.filepath)].sample(15, random_state=SEED)

d = qc.merge(splits[['filepath', 'split']], on='filepath')
d = d[(d.split == 'val') & ~d.qc_ok]
es_vacia = d.motivo.str.contains('vac', case=False, na=False)          # contrastar con el value_counts de arriba
desc = pd.concat([d[es_vacia].sample(4, random_state=SEED),             # 4 vacías
                  d[~es_vacia & (d.area_frac > 0.35)].sample(3, random_state=SEED)])   # 3 de área grande

muestra = pd.concat([azar.assign(estrato='aleatoria'), peor.assign(estrato='peor'),
                     desc.assign(estrato='descartada')])[['filepath', 'estrato']].reset_index(drop=True)
muestra.insert(0, 'id', range(1, len(muestra) + 1))                    # ids 1..30
assert muestra.filepath.is_unique and len(muestra) == 30
tmp = f'{ROOT}/sam2/muestra_val.csv.tmp'
muestra.to_csv(tmp, index=False); os.replace(tmp, f'{ROOT}/sam2/muestra_val.csv')
print(muestra.estrato.value_counts())

Ahora vamos a revisar que los archivos de salida quedaron donde los esperan los cuadernos B y C.

In [ ]:
for f in ['sam2/prompts.csv', 'sam2/muestra_val.csv']:
    print(f, 'OK' if os.path.exists(f'{ROOT}/{f}') else 'FALTA')
print(pd.read_csv(f'{ROOT}/sam2/muestra_val.csv').head(3))